# Phase 2 Review and Reporting Slice

This notebook proves candidate review state transitions, stronger provenance, and the minimal report surface.

In [1]:
from pathlib import Path
import sys
import tempfile

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from onto_canon6.pipeline import ReviewService  # noqa: E402
from onto_canon6.surfaces import ReviewReportService  # noqa: E402

In [2]:
tmp_dir = tempfile.TemporaryDirectory()
db_path = Path(tmp_dir.name) / "review.sqlite3"
service = ReviewService(db_path=db_path, default_acceptance_policy="record_only")
report_service = ReviewReportService(review_service=service)
db_path

PosixPath('/tmp/tmpgh_77gvp/review.sqlite3')

In [3]:
accepted_candidate = service.submit_candidate_assertion(
    payload={
        "predicate": "dodaf:activity_performs_resource",
        "roles": {
            "performer": [{"entity_id": "ent:performer:1", "entity_type": "dm2:Performer"}],
            "activity": [{"entity_id": "ent:activity:1", "entity_type": "dm2:OperationalActivity"}],
            "resource": [{"entity_id": "ent:resource:1", "entity_type": "dm2:Resource"}],
        },
    },
    profile_id="dodaf",
    profile_version="0.1.0",
    submitted_by="notebook:analyst",
    source_kind="notebook",
    source_ref="notebook://phase2/accepted-candidate",
    source_label="phase2 accepted candidate",
    source_metadata={"slice": "phase2", "case": "accepted_candidate"},
)
accepted_candidate.model_dump()

{'candidate': {'candidate_id': 'cand_2cd3a2ca820d48a086ee2b88',
  'profile': {'profile_id': 'dodaf', 'profile_version': '0.1.0'},
  'validation_status': 'valid',
  'review_status': 'pending_review',
  'payload_hash': 'sha256:f9dd398c9168dd46a47d08536a46135426c6838557ad108596fb81368526fd2b',
  'payload': {'predicate': 'dodaf:activity_performs_resource',
   'roles': {'activity': [{'entity_id': 'ent:activity:1',
      'entity_type': 'dm2:OperationalActivity',
      'kind': 'entity'}],
    'performer': [{'entity_id': 'ent:performer:1',
      'entity_type': 'dm2:Performer',
      'kind': 'entity'}],
    'resource': [{'entity_id': 'ent:resource:1',
      'entity_type': 'dm2:Resource',
      'kind': 'entity'}]}},
  'normalized_payload': {'predicate': 'dodaf:activity_performs_resource',
   'roles': {'activity': [{'entity_id': 'ent:activity:1',
      'entity_type': 'dm2:OperationalActivity',
      'kind': 'entity'}],
    'performer': [{'entity_id': 'ent:performer:1',
      'entity_type': 'dm2:P

In [4]:
pending_candidate = service.submit_candidate_assertion(
    payload={
        "predicate": "oc:unknown_predicate_demo",
        "roles": {
            "subject": [{"entity_id": "ent:subject:1", "entity_type": "oc:person"}],
        },
    },
    profile_id="psyop_seed",
    profile_version="0.1.0",
    submitted_by="notebook:analyst",
    source_kind="notebook",
    source_ref="notebook://phase2/pending-candidate",
    source_label="phase2 pending candidate",
    source_metadata={"slice": "phase2", "case": "pending_candidate"},
)
pending_candidate.model_dump()

{'candidate': {'candidate_id': 'cand_9d77d81be1cc47f0a454ec52',
  'profile': {'profile_id': 'psyop_seed', 'profile_version': '0.1.0'},
  'validation_status': 'needs_review',
  'review_status': 'pending_review',
  'payload_hash': 'sha256:a79b23766b1263e1d4b43058a110ab2731238253593a0835c3895d1610aee0fa',
  'payload': {'predicate': 'oc:unknown_predicate_demo',
   'roles': {'subject': [{'entity_id': 'ent:subject:1',
      'entity_type': 'oc:person',
      'kind': 'entity'}]}},
  'normalized_payload': {'predicate': 'oc:unknown_predicate_demo',
   'roles': {'subject': [{'entity_id': 'ent:subject:1',
      'entity_type': 'oc:person',
      'kind': 'entity'}]}},
  'validation': {'hard_errors': (),
   'soft_violations': ({'code': 'oc:profile_unknown_predicate',
     'message': "predicate 'oc:unknown_predicate_demo' is not declared in profile psyop_seed; mixed ontology mode routes it as a proposal",
     'path': 'predicate',
     'severity': 'soft'},),
   'type_checks_total': 0,
   'type_checks_

In [5]:
invalid_candidate = service.submit_candidate_assertion(
    payload={"roles": {}},
    profile_id="default",
    profile_version="1.0.0",
    submitted_by="notebook:analyst",
    source_kind="notebook",
    source_ref="notebook://phase2/invalid-candidate",
    source_label="phase2 invalid candidate",
    source_metadata={"slice": "phase2", "case": "invalid_candidate"},
)
invalid_candidate.model_dump()

{'candidate': {'candidate_id': 'cand_c706bfcbeb554dfd9e40e2df',
  'profile': {'profile_id': 'default', 'profile_version': '1.0.0'},
  'validation_status': 'invalid',
  'review_status': 'pending_review',
  'payload_hash': 'sha256:ec2f50c8eda21c1a5f4d8d9b7b806707d508dbfed460b8cbe6e0ef76c105a95c',
  'payload': {'roles': {}},
  'normalized_payload': {'roles': {}},
  'validation': {'hard_errors': ({'code': 'oc:hard_missing_predicate',
     'message': 'predicate must be a non-empty string',
     'path': 'predicate',
     'severity': 'hard'},
    {'code': 'oc:hard_missing_roles',
     'message': 'roles must be a non-empty object',
     'path': 'roles',
     'severity': 'hard'}),
   'soft_violations': (),
   'type_checks_total': 0,
   'type_checks_unknown': 0},
  'proposal_ids': (),
  'provenance': {'submitted_by': 'notebook:analyst',
   'source_artifact': {'source_kind': 'notebook',
    'source_ref': 'notebook://phase2/invalid-candidate',
    'source_label': 'phase2 invalid candidate',
    's

In [6]:
accepted_review = service.review_candidate(
    candidate_id=accepted_candidate.candidate.candidate_id,
    decision="accepted",
    actor_id="notebook:reviewer",
    note_text="This candidate is acceptable as-is.",
)
accepted_review.model_dump()

{'candidate_id': 'cand_2cd3a2ca820d48a086ee2b88',
 'profile': {'profile_id': 'dodaf', 'profile_version': '0.1.0'},
 'validation_status': 'valid',
 'review_status': 'accepted',
 'payload_hash': 'sha256:f9dd398c9168dd46a47d08536a46135426c6838557ad108596fb81368526fd2b',
 'payload': {'predicate': 'dodaf:activity_performs_resource',
  'roles': {'activity': [{'entity_id': 'ent:activity:1',
     'entity_type': 'dm2:OperationalActivity',
     'kind': 'entity'}],
   'performer': [{'entity_id': 'ent:performer:1',
     'entity_type': 'dm2:Performer',
     'kind': 'entity'}],
   'resource': [{'entity_id': 'ent:resource:1',
     'entity_type': 'dm2:Resource',
     'kind': 'entity'}]}},
 'normalized_payload': {'predicate': 'dodaf:activity_performs_resource',
  'roles': {'activity': [{'entity_id': 'ent:activity:1',
     'entity_type': 'dm2:OperationalActivity',
     'kind': 'entity'}],
   'performer': [{'entity_id': 'ent:performer:1',
     'entity_type': 'dm2:Performer',
     'kind': 'entity'}],
   '

In [7]:
rejected_review = service.review_candidate(
    candidate_id=invalid_candidate.candidate.candidate_id,
    decision="rejected",
    actor_id="notebook:reviewer",
    note_text="Invalid candidates can be rejected but not accepted.",
)
rejected_review.model_dump()

{'candidate_id': 'cand_c706bfcbeb554dfd9e40e2df',
 'profile': {'profile_id': 'default', 'profile_version': '1.0.0'},
 'validation_status': 'invalid',
 'review_status': 'rejected',
 'payload_hash': 'sha256:ec2f50c8eda21c1a5f4d8d9b7b806707d508dbfed460b8cbe6e0ef76c105a95c',
 'payload': {'roles': {}},
 'normalized_payload': {'roles': {}},
 'validation': {'hard_errors': ({'code': 'oc:hard_missing_predicate',
    'message': 'predicate must be a non-empty string',
    'path': 'predicate',
    'severity': 'hard'},
   {'code': 'oc:hard_missing_roles',
    'message': 'roles must be a non-empty object',
    'path': 'roles',
    'severity': 'hard'}),
  'soft_violations': (),
  'type_checks_total': 0,
  'type_checks_unknown': 0},
 'proposal_ids': (),
 'provenance': {'submitted_by': 'notebook:analyst',
  'source_artifact': {'source_kind': 'notebook',
   'source_ref': 'notebook://phase2/invalid-candidate',
   'source_label': 'phase2 invalid candidate',
   'source_metadata': {'case': 'invalid_candidat

In [8]:
pending_report = report_service.build_report(review_status_filter="pending_review")
accepted_report = report_service.build_report(review_status_filter="accepted")
{
    "pending_report": pending_report.model_dump(),
    "accepted_report": accepted_report.model_dump(),
}

{'pending_report': {'candidates': ({'candidate_id': 'cand_9d77d81be1cc47f0a454ec52',
    'profile': {'profile_id': 'psyop_seed', 'profile_version': '0.1.0'},
    'validation_status': 'needs_review',
    'review_status': 'pending_review',
    'payload_hash': 'sha256:a79b23766b1263e1d4b43058a110ab2731238253593a0835c3895d1610aee0fa',
    'payload': {'predicate': 'oc:unknown_predicate_demo',
     'roles': {'subject': [{'entity_id': 'ent:subject:1',
        'entity_type': 'oc:person',
        'kind': 'entity'}]}},
    'normalized_payload': {'predicate': 'oc:unknown_predicate_demo',
     'roles': {'subject': [{'entity_id': 'ent:subject:1',
        'entity_type': 'oc:person',
        'kind': 'entity'}]}},
    'validation': {'hard_errors': (),
     'soft_violations': ({'code': 'oc:profile_unknown_predicate',
       'message': "predicate 'oc:unknown_predicate_demo' is not declared in profile psyop_seed; mixed ontology mode routes it as a proposal",
       'path': 'predicate',
       'severity':

In [9]:
proposal_report = report_service.build_report(
    proposal_status_filter="pending",
    profile_id="psyop_seed",
    profile_version="0.1.0",
)
proposal_report.model_dump()

{'candidates': ({'candidate_id': 'cand_9d77d81be1cc47f0a454ec52',
   'profile': {'profile_id': 'psyop_seed', 'profile_version': '0.1.0'},
   'validation_status': 'needs_review',
   'review_status': 'pending_review',
   'payload_hash': 'sha256:a79b23766b1263e1d4b43058a110ab2731238253593a0835c3895d1610aee0fa',
   'payload': {'predicate': 'oc:unknown_predicate_demo',
    'roles': {'subject': [{'entity_id': 'ent:subject:1',
       'entity_type': 'oc:person',
       'kind': 'entity'}]}},
   'normalized_payload': {'predicate': 'oc:unknown_predicate_demo',
    'roles': {'subject': [{'entity_id': 'ent:subject:1',
       'entity_type': 'oc:person',
       'kind': 'entity'}]}},
   'validation': {'hard_errors': (),
    'soft_violations': ({'code': 'oc:profile_unknown_predicate',
      'message': "predicate 'oc:unknown_predicate_demo' is not declared in profile psyop_seed; mixed ontology mode routes it as a proposal",
      'path': 'predicate',
      'severity': 'soft'},),
    'type_checks_total':

In [10]:
try:
    service.review_candidate(
        candidate_id=invalid_candidate.candidate.candidate_id,
        decision="accepted",
        actor_id="notebook:other-reviewer",
    )
except Exception as exc:
    type(exc).__name__, str(exc)